In [ ]:
import cv2
import os
from pathlib import Path
import shutil
import random

In [2]:

# =========================
# CONFIG
# =========================
VIDEOS_DIR = "videos"                  # folder containing your videos
OUTPUT_ROOT = "extracted_frames"       # root folder for all extracted frames

SAVE_EVERY = 10                        # save 1 frame every 10 frames
VIDEO_EXTENSIONS = [".mp4", ".mov", ".avi", ".mkv"]

# =========================
# FUNCTION TO EXTRACT FRAMES
# =========================
def extract_frames_from_video(video_path, output_root, save_every=10):
    video_name = video_path.stem                      # filename without extension
    output_dir = Path(output_root) / video_name      # create subfolder for this video
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\nProcessing: {video_path.name}")
    print(f"FPS: {fps}")
    print(f"Total frames: {total_frames}")
    print(f"Saving into: {output_dir}")

    frame_id = 0
    saved_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % save_every == 0:
            save_path = output_dir / f"frame_{saved_id:05d}.jpg"
            cv2.imwrite(str(save_path), frame)
            saved_id += 1

        frame_id += 1

    cap.release()
    print(f"Saved {saved_id} frames for {video_path.name}")

# =========================
# MAIN LOOP OVER ALL VIDEOS
# =========================
videos_dir = Path(VIDEOS_DIR)
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

for video_path in videos_dir.iterdir():
    if video_path.is_file() and video_path.suffix.lower() in VIDEO_EXTENSIONS:
        extract_frames_from_video(video_path, OUTPUT_ROOT, SAVE_EVERY)

print("\nAll videos processed.")


Processing: Vid_1.mp4
FPS: 25.0
Total frames: 288
Saving into: extracted_frames\Vid_1
Saved 29 frames for Vid_1.mp4

All videos processed.


In [9]:
import shutil
import random
from pathlib import Path

# =========================
# CHECK CURRENT LOCATION
# =========================
cwd = Path.cwd()
print("Current working directory:", cwd)

# =========================
# AUTO-FIND IMAGE FOLDER
# =========================
possible_image_dirs = [
    cwd / "src" / "extracted_frames" / "Vid_1",
    cwd / "extracted_frames" / "Vid_1",
    cwd.parent / "src" / "extracted_frames" / "Vid_1",
]

images_src = None

for p in possible_image_dirs:
    if p.exists():
        images_src = p
        break

if images_src is None:
    print("Tried these paths:")
    for p in possible_image_dirs:
        print(p)
    raise FileNotFoundError("Could not find Vid_1 images folder.")

print("Images folder found:", images_src)

# =========================
# LABELS FOLDER
# =========================
labels_src = images_src / "Labels"

if not labels_src.exists():
    raise FileNotFoundError(f"Labels folder not found: {labels_src}")

print("Labels folder found:", labels_src)

# =========================
# DATASET ROOT
# =========================
# If notebook is inside src, dataset should be outside src
if cwd.name.lower() == "src":
    dataset_root = cwd.parent / "dataset"
else:
    dataset_root = cwd / "dataset"

print("Dataset root:", dataset_root)

# =========================
# CLEAN OLD DATASET FILES
# =========================
for split in ["train", "val", "test"]:
    img_dir = dataset_root / "images" / split
    lbl_dir = dataset_root / "labels" / split

    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for f in img_dir.glob("*"):
        f.unlink()

    for f in lbl_dir.glob("*"):
        f.unlink()

# =========================
# COLLECT LABELED IMAGES
# =========================
image_extensions = [".jpg", ".jpeg", ".png"]

labeled_images = []

for img_path in images_src.iterdir():
    if img_path.suffix.lower() in image_extensions:
        label_path = labels_src / f"{img_path.stem}.txt"

        if label_path.exists():
            labeled_images.append(img_path)
        else:
            print(f"Missing label for: {img_path.name}")

print(f"\nFound {len(labeled_images)} labeled images.")

if len(labeled_images) == 0:
    raise ValueError("No labeled images found. Check image and label paths.")

# =========================
# SPLIT DATA
# =========================
random.seed(42)
random.shuffle(labeled_images)

n = len(labeled_images)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

splits = {
    "train": labeled_images[:train_end],
    "val": labeled_images[train_end:val_end],
    "test": labeled_images[val_end:]
}

# =========================
# COPY IMAGES + LABELS
# =========================
for split, images in splits.items():
    for img_path in images:
        label_path = labels_src / f"{img_path.stem}.txt"

        dst_img = dataset_root / "images" / split / img_path.name
        dst_label = dataset_root / "labels" / split / label_path.name

        shutil.copy2(img_path, dst_img)
        shutil.copy2(label_path, dst_label)

    print(f"{split}: {len(images)} images")

print("\nDataset prepared successfully.")

Current working directory: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src
Images folder found: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1
Labels folder found: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\Labels
Dataset root: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\dataset

Found 16 labeled images.
train: 11 images
val: 3 images
test: 2 images

Dataset prepared successfully.


In [21]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection")
DATA_YAML = PROJECT_ROOT / "dataset" / "data.yaml"

print("Data yaml exists:", DATA_YAML.exists())
print("Data yaml path:", DATA_YAML)

model = YOLO("yolov8n.pt")

model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=2,
    device="cpu",
    amp=False,
    workers=0,
    name="flow_test_cpu"
)

Data yaml exists: True
Data yaml path: C:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\dataset\data.yaml
Ultralytics 8.4.46  Python-3.11.1 torch-2.11.0+cpu CPU (Intel Core i5-10300H 2.50GHz)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021924EC8290>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [26]:
from ultralytics import YOLO

model = YOLO(r"runs/detect/flow_test_cpu-4/weights/best.pt")

model.predict(
    source=r"extracted_frames/Vid_1",
    save=True,
    save_txt=True,
    conf=0.25,
    device="cpu"
)


image 1/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00016.jpg: 384x640 1 Giraffe, 54.9ms
image 2/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00017.jpg: 384x640 1 Giraffe, 46.3ms
image 3/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00018.jpg: 384x640 1 Giraffe, 44.8ms
image 4/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00019.jpg: 384x640 1 Giraffe, 46.6ms
image 5/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00020.jpg: 384x640 1 Giraffe, 41.9ms
image 6/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00021.jpg: 384x640 1 Giraffe, 44.0ms
image 7/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bo

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Giraffe'}
 obb: None
 orig_img: array([[[221, 195, 158],
         [221, 195, 158],
         [221, 195, 158],
         ...,
         [232, 225, 186],
         [232, 225, 186],
         [232, 225, 186]],
 
        [[221, 195, 158],
         [221, 195, 158],
         [221, 195, 158],
         ...,
         [232, 225, 186],
         [232, 225, 186],
         [232, 225, 186]],
 
        [[221, 195, 158],
         [221, 195, 158],
         [221, 195, 158],
         ...,
         [232, 225, 186],
         [232, 225, 186],
         [232, 225, 186]],
 
        ...,
 
        [[ 78,  99, 130],
         [ 78,  99, 130],
         [ 78,  99, 130],
         ...,
         [ 73, 104, 129],
         [ 74, 104, 129],
         [ 74, 104, 129]],
 
        [[ 78,  99, 130],
         [ 78,  99, 130],
         [ 78,  99, 130],
         ...,
         [ 74, 10